In [29]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
import logging

# ==========================================
# Global configuration (no fixed seed – each run produces different samples)
# ==========================================

TARGET_LEN = 6000
INPUT_CHANNEL = 3
FS = 100.0

OUTPUT_DIR = "./outputs_selected_waveforms_no_prob"
os.makedirs(OUTPUT_DIR, exist_ok=True)

REAL_NOISE_PATH = "./stead_real_noise_50000.npy"
METADATA_PATH = "./metadata.csv"

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.dpi'] = 300
FIXED_YLIM = [-15, 15]

# ==========================================
# Helper functions
# ==========================================
def normalize_single_sample(x):
    x_normalized = np.zeros_like(x, dtype=np.float32)
    for ch in range(INPUT_CHANNEL):
        mean = np.mean(x[:, ch])
        std = np.std(x[:, ch])
        if std < 1e-6:
            std = 1.0
        x_normalized[:, ch] = (x[:, ch] - mean) / std
    return x_normalized

def generate_triangle_label(length, peak_idx, width=40):
    label = np.zeros(length, dtype=np.float32)
    half_width = width // 2
    start = max(0, peak_idx - half_width)
    end = min(length, peak_idx + half_width)
    rise_len = peak_idx - start
    if rise_len > 0:
        label[start:peak_idx] = np.linspace(0, 1, rise_len)
    label[peak_idx] = 1.0
    fall_len = end - peak_idx
    if fall_len > 0:
        label[peak_idx+1:end] = np.linspace(1, 0, fall_len-1)
    return label

def generate_3channel_labels(y_2channel, width_p=20, width_s=40):
    n_samples, n_timesteps = y_2channel.shape[0], y_2channel.shape[1]
    y_3channel = np.zeros((n_samples, n_timesteps, 3), dtype=np.float32)
    for i in range(n_samples):
        p_idx = np.argmax(y_2channel[i, :, 0])
        s_idx = np.argmax(y_2channel[i, :, 1])
        y_3channel[i, :, 0] = generate_triangle_label(n_timesteps, p_idx, width=width_p)
        y_3channel[i, :, 1] = generate_triangle_label(n_timesteps, s_idx, width=width_s)
    y_3channel[..., 2] = 1.0 - y_3channel[..., 0] - y_3channel[..., 1]
    return np.clip(y_3channel, 0.0, 1.0)

def calculate_snr(waveform, p_idx, s_idx):
    signal_start = max(0, p_idx - 100)
    signal_end = min(len(waveform), s_idx + 200)
    noise_start = max(0, signal_start - 500)
    noise_end = signal_start
    if noise_end <= noise_start:
        return 0.0
    signal_power = np.mean(waveform[signal_start:signal_end, 2] ** 2)
    noise_power = np.mean(waveform[noise_start:noise_end, 2] ** 2)
    if noise_power < 1e-10:
        return 100.0
    return 10 * np.log10(signal_power / noise_power)

# ==========================================
# Plotting function (3-component waveforms only)
# ==========================================
def plot_waveform_only(sample_idx, raw_wave, true_label, magnitude, snr, save_path,
                       category_name):
    """
    Plot three-component waveforms with true P/S picks (if provided).
    """
    t = np.arange(TARGET_LEN)
    comp_names = ['E', 'N', 'Z']
    comp_color = '#1f77b4'
    color_p_true = '#cccc00'
    color_s_true = '#ff0000'

    if true_label is not None and true_label.ndim == 2 and true_label.shape[1] >= 2:
        tp_true = np.argmax(true_label[:, 0])
        ts_true = np.argmax(true_label[:, 1])
    else:
        tp_true = None
        ts_true = None

    if tp_true is not None and ts_true is not None:
        x_lim_start = max(0, tp_true - 1000)
        x_lim_end = min(TARGET_LEN, ts_true + 1000)
    else:
        x_lim_start = 0
        x_lim_end = TARGET_LEN

    fig = plt.figure(figsize=(12, 8), dpi=300)
    gs = fig.add_gridspec(4, 1, height_ratios=[0.6, 1, 1, 1], hspace=0.12)

    # Title
    ax_title = fig.add_subplot(gs[0, 0])
    ax_title.axis('off')
    if category_name == 'noise':
        title_text = 'Pure Noise Sample'
    else:
        title_text = f'M={magnitude:.1f}, SNR={snr:.1f} dB'
    ax_title.text(0.5, 0.5, title_text, ha='center', va='center',
                  fontsize=14, fontweight='bold')

    # Waveforms
    norm_wave = normalize_single_sample(raw_wave)
    for i in range(3):
        ax = fig.add_subplot(gs[i+1, 0])
        ax.plot(t, norm_wave[:, i], color=comp_color, linewidth=0.8, label=comp_names[i])
        if tp_true is not None:
            ax.axvline(tp_true, color=color_p_true, linestyle='-', linewidth=1.5, label='P Manual')
        if ts_true is not None:
            ax.axvline(ts_true, color=color_s_true, linestyle='-', linewidth=1.5, label='S Manual')
        ax.set_ylabel('Amplitude', fontsize=11)
        ax.set_ylim(FIXED_YLIM)
        ax.set_yticks([-15, 0, 15])
        ax.set_yticklabels(['-1', '0', '1'])
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(alpha=0.2)
        ax.set_xlim([x_lim_start, x_lim_end])
        if i < 2:
            ax.set_xticks([])
        else:
            ax.set_xlabel('Samples', fontsize=11)

    plt.savefig(save_path, bbox_inches='tight')
    plt.close()

# ==========================================
# Data loading (fixed test split using local random state)
# ==========================================
def load_data():
    logger.info("Loading STEAD test set and real noise...")
    X_orig = np.load("chunk2_stead_X.npy").astype(np.float32)
    y_2ch = np.load("chunk2_stead_y.npy").astype(np.float32)
    y_3ch = generate_3channel_labels(y_2ch, width_p=20, width_s=40)

    metadata = pd.read_csv(METADATA_PATH, low_memory=False)
    magnitudes = metadata['source_magnitude'].values.astype(np.float32)

    # Use a local random state to keep the test split constant across runs
    rng = np.random.RandomState(42)
    indices = np.arange(len(X_orig))
    rng.shuffle(indices)
    n_total = len(X_orig)
    n_train = int(n_total * 0.7)
    n_val = int(n_total * 0.2)
    test_idx = indices[n_train + n_val:]

    X_test = X_orig[test_idx]
    y_test = y_3ch[test_idx]
    test_mag = magnitudes[test_idx]

    # Load real noise
    logger.info("Loading real noise data...")
    if not os.path.exists(REAL_NOISE_PATH):
        raise FileNotFoundError(f"Real noise file not found: {REAL_NOISE_PATH}")
    noise_data = np.load(REAL_NOISE_PATH)
    if noise_data.ndim == 2:
        noise_data = np.stack([noise_data] * 3, axis=-1)
    elif noise_data.ndim == 3 and noise_data.shape[2] != 3:
        noise_data = noise_data[:, :, :3]
    logger.info(f"Noise data shape: {noise_data.shape}")

    return X_test, y_test, test_mag, noise_data

# ==========================================
# Sample selection (random each run)
# ==========================================
def select_one_sample(X_test, y_test, magnitudes, snr_list, low, high, category):
    candidates = []
    for i in range(len(X_test)):
        if low <= magnitudes[i] <= high:
            if (category == 'low_snr' and 5.0 <= snr_list[i] <= 10.0) or (category != 'low_snr'):
                candidates.append(i)
    if not candidates:
        if category == 'large_mag':
            candidates = [i for i in range(len(X_test)) if magnitudes[i] >= 4.0]
        elif category == 'small_mag':
            candidates = [i for i in range(len(X_test)) if magnitudes[i] <= 2.0]
        elif category == 'low_snr':
            candidates = [i for i in range(len(X_test)) if snr_list[i] < 15.0]
    if not candidates:
        raise RuntimeError(f"No samples for category '{category}'")
    return np.random.choice(candidates)

# ==========================================
# Main
# ==========================================
def main():
    X_test, y_test, test_mag, noise_data = load_data()

    # Compute SNRs for all test events
    snr_list = []
    for i in range(len(X_test)):
        p_idx = np.argmax(y_test[i, :, 0])
        s_idx = np.argmax(y_test[i, :, 1])
        snr_list.append(calculate_snr(X_test[i], p_idx, s_idx))
    snr_list = np.array(snr_list)

    # Select one sample per category (different every run)
    large_idx = select_one_sample(X_test, y_test, test_mag, snr_list, 4.0, 5.0, 'large_mag')
    small_idx = select_one_sample(X_test, y_test, test_mag, snr_list, 0.1, 1.0, 'small_mag')
    low_snr_idx = select_one_sample(X_test, y_test, test_mag, snr_list, 0.0, 10.0, 'low_snr')
    if low_snr_idx is None:
        low_snr_idx = np.argmin(snr_list)

    samples = [
        ('large_mag', large_idx),
        ('small_mag', small_idx),
        ('low_snr', low_snr_idx)
    ]

    for cat, idx in samples:
        raw = X_test[idx]
        label = y_test[idx]
        mag = test_mag[idx]
        snr = snr_list[idx]
        save_path = os.path.join(OUTPUT_DIR, f"{cat}_sample_idx{idx}.png")
        plot_waveform_only(idx, raw, label, mag, snr, save_path, category_name=cat)

    # Pure noise sample
    noise_idx = np.random.randint(0, len(noise_data))
    noise_wave = noise_data[noise_idx]
    if noise_wave.shape[0] < TARGET_LEN:
        repeats = int(np.ceil(TARGET_LEN / noise_wave.shape[0]))
        noise_wave = np.tile(noise_wave, (repeats, 1))[:TARGET_LEN]
    else:
        noise_wave = noise_wave[:TARGET_LEN]
    save_path = os.path.join(OUTPUT_DIR, "pure_noise_sample.png")
    plot_waveform_only(0, noise_wave, None, None, None, save_path, category_name='noise')

    logger.info(f"All 4 waveform figures saved to: {OUTPUT_DIR}")

if __name__ == "__main__":
    main()

2026-06-05 01:54:12,533 - Loading STEAD test set and real noise...
2026-06-05 01:54:40,995 - Loading real noise data...
2026-06-05 01:54:41,747 - Noise data shape: (50000, 6000, 3)
2026-06-05 01:54:43,871 - All 4 waveform figures saved to: ./outputs_selected_waveforms_no_prob


In [33]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import logging

# ===================== Configuration =====================
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.dpi'] = 300

OUTPUT_DIR = "./outputs_dataset_distributions"   # new folder
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===================== Helper functions =====================
def calculate_snr(waveform, p_idx, s_idx):
    """Compute local SNR on the Z component."""
    signal_start = max(0, p_idx - 100)
    signal_end = min(len(waveform), s_idx + 200)
    noise_start = max(0, signal_start - 500)
    noise_end = signal_start
    if noise_end <= noise_start:
        return 0.0
    signal_power = np.mean(waveform[signal_start:signal_end, 2] ** 2)
    noise_power = np.mean(waveform[noise_start:noise_end, 2] ** 2)
    if noise_power < 1e-10:
        return 100.0
    return 10 * np.log10(signal_power / noise_power)

def generate_triangle_label(length, peak_idx, width=40):
    """Generate a single-channel triangular label."""
    label = np.zeros(length, dtype=np.float32)
    half_width = width // 2
    start = max(0, peak_idx - half_width)
    end = min(length, peak_idx + half_width)
    rise_len = peak_idx - start
    if rise_len > 0:
        label[start:peak_idx] = np.linspace(0, 1, rise_len)
    label[peak_idx] = 1.0
    fall_len = end - peak_idx
    if fall_len > 0:
        label[peak_idx+1:end] = np.linspace(1, 0, fall_len-1)
    return label

def generate_3channel_labels(y_2channel, width_p=20, width_s=40):
    """Convert 2‑channel labels to 3‑channel (P, S, Noise)."""
    n_samples, n_timesteps = y_2channel.shape[0], y_2channel.shape[1]
    y_3channel = np.zeros((n_samples, n_timesteps, 3), dtype=np.float32)
    for i in range(n_samples):
        p_idx = np.argmax(y_2channel[i, :, 0])
        s_idx = np.argmax(y_2channel[i, :, 1])
        y_3channel[i, :, 0] = generate_triangle_label(n_timesteps, p_idx, width=width_p)
        y_3channel[i, :, 1] = generate_triangle_label(n_timesteps, s_idx, width=width_s)
    y_3channel[..., 2] = 1.0 - y_3channel[..., 0] - y_3channel[..., 1]
    return np.clip(y_3channel, 0.0, 1.0)

# ===================== Main =====================
def main():
    logger.info("Loading full chunk2 dataset...")
    X = np.load("chunk2_stead_X.npy").astype(np.float32)
    y_2ch = np.load("chunk2_stead_y.npy").astype(np.float32)
    y_3ch = generate_3channel_labels(y_2ch)   # used only to obtain P/S indices

    metadata = pd.read_csv("metadata.csv", low_memory=False)
    magnitudes = metadata['source_magnitude'].values.astype(np.float32)

    # Compute SNR for all samples
    snr_list = []
    logger.info("Computing SNRs (may take a few minutes)...")
    for i in range(len(X)):
        p_idx = np.argmax(y_3ch[i, :, 0])
        s_idx = np.argmax(y_3ch[i, :, 1])
        snr = calculate_snr(X[i], p_idx, s_idx)
        snr_list.append(snr)
    snr_array = np.array(snr_list)

    # Define bins
    mag_bins = np.arange(0, 5.01, 0.5)      # 0, 0.5, ..., 5.0
    snr_bins = np.arange(0, 60.01, 5)        # 0, 5, ..., 60

    # Filter data within bin range
    mag_mask = (magnitudes >= mag_bins[0]) & (magnitudes <= mag_bins[-1])
    snr_mask = (snr_array >= snr_bins[0]) & (snr_array <= snr_bins[-1])

    mag_plot = magnitudes[mag_mask]
    snr_plot = snr_array[snr_mask]

    # Plot histograms
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(mag_plot, bins=mag_bins, color='steelblue', edgecolor='white', alpha=0.8)
    axes[0].set_xlabel('Magnitude')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Magnitude Distribution')
    axes[0].set_xticks(mag_bins)
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(True, alpha=0.3)

    axes[1].hist(snr_plot, bins=snr_bins, color='darkorange', edgecolor='white', alpha=0.8)
    axes[1].set_xlabel('SNR (dB)')
    axes[1].set_ylabel('Count')
    axes[1].set_title('SNR Distribution')
    axes[1].set_xticks(snr_bins)
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, "magnitude_snr_histograms.png")
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()
    logger.info(f"Histograms saved to: {save_path}")

if __name__ == "__main__":
    main()

2026-06-05 15:57:27,941 - Loading full chunk2 dataset...
2026-06-05 15:57:56,534 - Computing SNRs (may take a few minutes)...
2026-06-05 15:58:02,823 - Histograms saved to: ./outputs_dataset_distributions/magnitude_snr_histograms.png
